# Notebook 07 - Professor Demo (Types 1-7)

This notebook is a presentation-facing walkthrough of the final sanctions IR system.

- For query types **1-6**, the primary output is **RRF** over the retrieval lanes.
- For query type **7**, the primary output is **RAG** over top-k fused retrieval context.
- All demo examples come from the **prepared query files** under `data/queries/`.
- We run **one fixed query per type** for the live demo and keep a few backup query IDs in comments in the config cell.

The goal here is clarity, not full evaluation: short tables, one representative query per type, and an easy structure to present live.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 50)


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "data" / "queries").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current working directory.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TOP_K = 5
RETRIEVER_K = 25
RRF_K = 60
DENSE_RUN_TAGS = {"minilm": "FULL_20260409", "bge-m3": "FULL_20260410"}  # Per-model collection tags.
RAG_RUN_TAG = "CURRENT"        # Change this if your Type 7 outputs use a different tag.

DOCS_PATH = ROOT / "data" / "json_format_data" / "subset" / "documents.jsonl"
MODELS_DIR = ROOT / "models"
CHROMA_DIR = ROOT / "chroma_db"
QUERIES_DIR = ROOT / "data" / "queries"
RESULTS_DIR = ROOT / "results"
RAG_DIR = RESULTS_DIR / "rag"
RAG_EVAL_DIR = RAG_DIR / "evaluation" / RAG_RUN_TAG

DEMO_QUERY_IDS = {
    1: "Q1_022",  # IMO9553359 -> DONG CHANG [Vessel]
    2: "Q2_001",  # Yatai Smart Industrial New City
    3: "Q3_001",  # Russian oil tankers in the shadow fleet...
    4: "Q4_001",  # What vessels does Sovcomflot own or manage?
    5: "Q5_008",  # Mahan Airways
    6: "Q6_010",  # CA SEMA person Russia
    7: "Q7_001",  # Type 7 narrative version of Q3_001
}

# Backup prepared query IDs if you want to switch live during the presentation:
# Type 1: Q1_022, Q1_023, Q1_024
# Type 2: Q2_001, Q2_007, Q2_008
# Type 3: Q3_001, Q3_006, Q3_011
# Type 4: Q4_001, Q4_002, Q4_008
# Type 5: Q5_008, Q5_001, Q5_006
# Type 6: Q6_010, Q6_011, Q6_007
# Type 7: Q7_001, Q7_006, Q7_027

In [2]:
QUERY_FILES = {
    1: QUERIES_DIR / "queries_type_1.json",
    2: QUERIES_DIR / "queries_type_2.json",
    3: QUERIES_DIR / "queries_type_3.json",
    4: QUERIES_DIR / "queries_type_4.json",
    5: QUERIES_DIR / "queries_type_5.json",
    6: QUERIES_DIR / "queries_type_6.json",
    7: QUERIES_DIR / "type7_queries.json",
}


def load_query_record(query_type: int, query_id: str) -> dict:
    path = QUERY_FILES[query_type]
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    for item in payload.get("queries", []):
        if str(item.get("query_id")) != str(query_id):
            continue

        record = dict(item)
        record["query_id"] = str(record["query_id"])
        record["query_type"] = int(record.get("query_type", query_type))
        if query_type == 7:
            record["query_text"] = str(record.get("query_text", ""))
            record["notes"] = str(record.get("query_notes", ""))
        else:
            texts = record.get("query_texts") or []
            record["query_text"] = str(texts[0]) if texts else ""
            record["notes"] = str(record.get("notes", ""))
        return record

    raise KeyError(f"Query {query_id} not found in {path.name}")


def build_selected_queries() -> pd.DataFrame:
    rows = []
    for query_type, query_id in DEMO_QUERY_IDS.items():
        record = load_query_record(query_type, query_id)
        rows.append(
            {
                "query_type": query_type,
                "query_id": record["query_id"],
                "query_text": record["query_text"],
                "notes": record.get("notes", ""),
                "filter_criteria": record.get("filter_criteria", ""),
                "source_query_id": record.get("source_query_id", ""),
            }
        )
    return pd.DataFrame(rows).sort_values("query_type").reset_index(drop=True)


def artifact_status() -> pd.DataFrame:
    rows = [
        {"artifact": "documents", "path": DOCS_PATH, "exists": DOCS_PATH.exists()},
        {"artifact": "bm25 index", "path": MODELS_DIR / "bm25" / "index.pkl", "exists": (MODELS_DIR / "bm25" / "index.pkl").exists()},
        {"artifact": "tfidf vectorizer", "path": MODELS_DIR / "tfidf" / "vectorizer.pkl", "exists": (MODELS_DIR / "tfidf" / "vectorizer.pkl").exists()},
        {"artifact": "identifier index", "path": MODELS_DIR / "identifier" / "index.pkl", "exists": (MODELS_DIR / "identifier" / "index.pkl").exists()},
        {"artifact": "chroma db", "path": CHROMA_DIR, "exists": CHROMA_DIR.exists()},
        {"artifact": "type7 context", "path": RAG_DIR / f"context_{RAG_RUN_TAG}.jsonl", "exists": (RAG_DIR / f"context_{RAG_RUN_TAG}.jsonl").exists()},
        {"artifact": "type7 answers", "path": RAG_DIR / f"answers_{RAG_RUN_TAG}.jsonl", "exists": (RAG_DIR / f"answers_{RAG_RUN_TAG}.jsonl").exists()},
        {"artifact": "type7 per_query", "path": RAG_EVAL_DIR / "per_query.csv", "exists": (RAG_EVAL_DIR / "per_query.csv").exists()},
        {"artifact": "type7 summary", "path": RAG_EVAL_DIR / "summary.json", "exists": (RAG_EVAL_DIR / "summary.json").exists()},
    ]
    df = pd.DataFrame(rows)
    df["path"] = df["path"].map(str)
    return df


selected_queries_df = build_selected_queries()
display(Markdown("## Selected Demo Queries"))
display(selected_queries_df)

display(Markdown("## Artifact Check"))
display(artifact_status())

## Selected Demo Queries

,query_type,query_id,query_text,notes,filter_criteria,source_query_id
0,1,Q1_022,IMO9553359,imoNumber | DONG CHANG [Vessel],,
1,2,Q2_001,Yatai Smart Industrial New City,"alias | caption: ""Myanmar Yatai International Holding Group Co., LTD."" [Company]",,
2,3,Q3_001,Russian oil tankers in the shadow fleet sanctioned for circumventing the price cap,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,,
3,4,Q4_001,What vessels does Sovcomflot own or manage?,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,,
4,5,Q5_008,Mahan Airways,35 doc_ids for same caption [LegalEntity],,
5,6,Q6_010,CA SEMA person Russia,1790 matching docs,programId=CA-SEMA | schema=Person | country=ru,
6,7,Q7_001,"Summarise the sanctions-related background, key entities, and supporting evidence for: Russian oil tankers in the shadow fleet sanctioned for circumventing ...",SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,,Q3_001


## Artifact Check

,artifact,path,exists
0,documents,/home/gfe/IR_project/data/json_format_data/subset/documents.jsonl,True
1,bm25 index,/home/gfe/IR_project/models/bm25/index.pkl,True
2,tfidf vectorizer,/home/gfe/IR_project/models/tfidf/vectorizer.pkl,True
3,identifier index,/home/gfe/IR_project/models/identifier/index.pkl,True
4,chroma db,/home/gfe/IR_project/chroma_db,True
5,type7 context,/home/gfe/IR_project/results/rag/context_CURRENT.jsonl,True
6,type7 answers,/home/gfe/IR_project/results/rag/answers_CURRENT.jsonl,True
7,type7 per_query,/home/gfe/IR_project/results/rag/evaluation/CURRENT/per_query.csv,True
8,type7 summary,/home/gfe/IR_project/results/rag/evaluation/CURRENT/summary.json,True


## Types 1-6: RRF Demo

This section runs the retrieval system live for one prepared query per type.

- We still show the component retrievers for interpretation.
- The main result to present is the **RRF** table.
- If an artifact is missing on the current machine, the setup cell above will show it before this section fails.

In [3]:
from src.fusion.rrf import ReciprocalRankFusion
from src.preprocessing.text_processing import TextProcessor
from src.rag.context_builder import load_document_lookup
from src.retrieval.classical_ir import BM25Retriever, IdentifierRetriever, TFIDFRetriever
from src.retrieval.dense_retriever import DenseRetriever


def require_paths(paths: list[Path]) -> None:
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required retrieval artifacts for the live demo:\n- " + "\n- ".join(missing)
        )


require_paths(
    [
        DOCS_PATH,
        MODELS_DIR / "bm25" / "index.pkl",
        MODELS_DIR / "tfidf" / "vectorizer.pkl",
        MODELS_DIR / "identifier" / "index.pkl",
        CHROMA_DIR,
    ]
)

tp = TextProcessor()
rrf = ReciprocalRankFusion(k=RRF_K)

bm25 = BM25Retriever()
bm25.load(MODELS_DIR / "bm25")

tfidf = TFIDFRetriever()
tfidf.load(MODELS_DIR / "tfidf")

identifier = IdentifierRetriever()
identifier.load(MODELS_DIR / "identifier" / "index.pkl")

dense_retrievers: dict[str, DenseRetriever] = {}
for model_key, run_tag in DENSE_RUN_TAGS.items():
    try:
        dense_retrievers[model_key] = DenseRetriever(
            model_key=model_key,
            run_tag=run_tag,
            chroma_dir=CHROMA_DIR,
            models_dir=MODELS_DIR,
        )
        print(f"Dense retriever ready: {model_key} ({run_tag})")
    except Exception as exc:
        print(f"Skipping dense retriever {model_key}: {exc}")


def normalised_query_text(query_text: str) -> str:
    return tp.normalize(query_text)


def bm25_query_tokens(query_text: str) -> list[str]:
    return normalised_query_text(query_text).split()


def project_dense_hits(hits: list[tuple[str, float, dict]]) -> list[tuple[str, float]]:
    return [(doc_id, float(score)) for doc_id, score, _meta in hits]


def run_query_across_models(query_record: dict) -> dict:
    query_text = str(query_record["query_text"])
    query_type = int(query_record["query_type"])

    component_hits: dict[str, list[tuple[str, float]]] = {}

    identifier_hits = identifier.search(query_text)
    if identifier_hits:
        component_hits["Identifier"] = identifier_hits[:RETRIEVER_K]

    component_hits["BM25"] = bm25.search(bm25_query_tokens(query_text), k=RETRIEVER_K)
    component_hits["TF-IDF"] = tfidf.search(normalised_query_text(query_text), k=RETRIEVER_K)

    dense_specs = [
        ("minilm", "Dense MiniLM"),
        ("bge-m3", "Dense BGE-M3"),
    ]
    for model_key, label in dense_specs:
        retriever = dense_retrievers.get(model_key)
        if retriever is None:
            continue
        try:
            component_hits[label] = project_dense_hits(retriever.search(query_text, k=RETRIEVER_K))
        except Exception as exc:
            print(f"Skipping {label} for {query_record['query_id']}: {exc}")

    ranked_lists = []
    for hits in component_hits.values():
        ranked_lists.append([(doc_id, rank) for rank, (doc_id, _score) in enumerate(hits, start=1)])
    rrf_hits = rrf.fuse(*ranked_lists) if ranked_lists else []

    return {
        "query_type": query_type,
        "query_id": query_record["query_id"],
        "query_text": query_text,
        "notes": query_record.get("notes", ""),
        "filter_criteria": query_record.get("filter_criteria", ""),
        "component_hits": component_hits,
        "rrf_hits": rrf_hits[:RETRIEVER_K],
    }


def collect_needed_doc_ids(search_outputs: dict[int, dict]) -> set[str]:
    needed: set[str] = set()
    for bundle in search_outputs.values():
        for hits in bundle["component_hits"].values():
            needed.update(doc_id for doc_id, _score in hits[:TOP_K])
        needed.update(doc_id for doc_id, _score in bundle["rrf_hits"][:TOP_K])
    return needed


def build_rrf_table(bundle: dict, docs_lookup: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for rank, (doc_id, score) in enumerate(bundle["rrf_hits"][:TOP_K], start=1):
        meta = docs_lookup.get(doc_id, {})
        rows.append(
            {
                "rank": rank,
                "doc_id": doc_id,
                "caption": meta.get("caption", ""),
                "schema": meta.get("schema", ""),
                "programs": meta.get("meta_programId", ""),
                "rrf_score": round(float(score), 6),
            }
        )
    return pd.DataFrame(rows)


def build_component_summary(bundle: dict, docs_lookup: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for lane, hits in bundle["component_hits"].items():
        if not hits:
            continue
        doc_id, score = hits[0]
        meta = docs_lookup.get(doc_id, {})
        rows.append(
            {
                "lane": lane,
                "top_doc_id": doc_id,
                "top_caption": meta.get("caption", ""),
                "top_schema": meta.get("schema", ""),
                "top_score": round(float(score), 6),
            }
        )
    return pd.DataFrame(rows)


def explain_type(query_type: int) -> str:
    return {
        1: "Exact identifier lookup - Identifier retriever should dominate, with RRF used as the presentation output.",
        2: "Name or alias search - lexical matching usually stays strong, while fusion shows how the full system behaves.",
        3: "Semantic/descriptive retrieval - dense search becomes more useful because vocabulary overlap can be weak.",
        4: "Relational/graph-style retrieval - RRF is useful because different retrievers surface different linked entities.",
        5: "Cross-dataset duplication - the demo should show multiple records for the same entity caption.",
        6: "Jurisdiction/filter-style retrieval - this is the hardest type and a good reminder that some needs are really structured filtering tasks.",
    }[query_type]


search_outputs = {
    query_type: run_query_across_models(load_query_record(query_type, DEMO_QUERY_IDS[query_type]))
    for query_type in range(1, 7)
}

needed_doc_ids = collect_needed_doc_ids(search_outputs)
docs_lookup = load_document_lookup(DOCS_PATH, needed_doc_ids)

for query_type in range(1, 7):
    bundle = search_outputs[query_type]
    display(Markdown(f"### Type {query_type} - {bundle['query_id']}"))
    display(Markdown(f"**Query:** {bundle['query_text']}"))
    if bundle.get("notes"):
        display(Markdown(f"**Notes:** {bundle['notes']}"))
    if bundle.get("filter_criteria"):
        display(Markdown(f"**Filter criteria:** {bundle['filter_criteria']}"))
    display(Markdown(explain_type(query_type)))

    display(Markdown("**Primary output: RRF**"))
    display(build_rrf_table(bundle, docs_lookup))

    display(Markdown("**Component retrievers (top hit only)**"))
    display(build_component_summary(bundle, docs_lookup))

[BM25] Loaded index with 1,249,379 documents


[TFIDF] Loaded matrix (1249379, 398914) with 1,249,379 documents
[Identifier] Loaded index with 560,373 keys
Dense retriever ready: minilm (FULL_20260409)
Dense retriever ready: bge-m3 (FULL_20260410)


### Type 1 - Q1_022

**Query:** IMO9553359

**Notes:** imoNumber | DONG CHANG [Vessel]

...

Jurisdiction/filter-style retrieval - this is the hardest type and a good reminder that some needs are really structured filtering tasks.

**Primary output: RRF**

,rank,doc_id,caption,schema,programs,rrf_score
0,1,Q94611309,H.K. Sema,Person,,0.032522
1,2,lt-pinreg-52a280a9c03e92d0b4465741de270b397e5e9cad,TADAS ŠĖMA,Person,,0.032266
2,3,NK-dTmDgWBG87Dnvrx5FmeGoa,Sema SİLKİN ÜN,Person,,0.032002
3,4,Q6059054,Sema Pişkinsüt,Person,,0.031010
4,5,Q130932120,Sukhato A. Sema,Person,,0.030550


**Component retrievers (top hit only)**

,lane,top_doc_id,top_caption,top_schema,top_score
0,BM25,Q94611309,H.K. Sema,Person,18.622626
1,TF-IDF,lt-pinreg-52a280a9c03e92d0b4465741de270b397e5e9cad,TADAS ŠĖMA,Person,0.568434
2,Dense MiniLM,ru-inn-773118290455,СВЕТЛАНА СЕРГЕЕВНА ЗЕНКЕВИЧ,Person,0.592728
3,Dense BGE-M3,Q120854310,Kamenskij Semen Sergeevich,Person,0.583785


## Type 7: RAG Demo

This section uses the prepared Type 7 query and the saved RAG outputs.

- The main thing to present is the grounded **answer**.
- The supporting evidence is the top retrieved **context rows**.
- If the Type 7 files are missing, update `RAG_RUN_TAG` in the config cell or generate the artifacts first.

In [4]:
CONTEXT_PATH = RAG_DIR / f"context_{RAG_RUN_TAG}.jsonl"
ANSWERS_PATH = RAG_DIR / f"answers_{RAG_RUN_TAG}.jsonl"
PER_QUERY_PATH = RAG_EVAL_DIR / "per_query.csv"
SUMMARY_PATH = RAG_EVAL_DIR / "summary.json"


def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


missing_type7 = [
    path for path in [CONTEXT_PATH, ANSWERS_PATH, PER_QUERY_PATH, SUMMARY_PATH] if not path.exists()
]

if missing_type7:
    display(Markdown("### Type 7 artifacts are missing"))
    print("Update RAG_RUN_TAG or generate the saved Type 7 outputs first:")
    for path in missing_type7:
        print(f"- {path}")
else:
    type7_record = load_query_record(7, DEMO_QUERY_IDS[7])
    context_df = load_jsonl(CONTEXT_PATH)
    answers_df = load_jsonl(ANSWERS_PATH)
    per_query_df = pd.read_csv(PER_QUERY_PATH)
    with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
        summary = json.load(handle)

    selected_context = (
        context_df[context_df["query_id"].astype(str) == type7_record["query_id"]]
        .sort_values("rank")
        .head(TOP_K)
        .copy()
    )
    selected_answer = answers_df[answers_df["query_id"].astype(str) == type7_record["query_id"]].copy()
    selected_metrics = per_query_df[per_query_df["query_id"].astype(str) == type7_record["query_id"]].copy()

    display(Markdown(f"### Type 7 - {type7_record['query_id']}"))
    display(Markdown(f"**Query:** {type7_record['query_text']}"))
    if type7_record.get("notes"):
        display(Markdown(f"**Notes:** {type7_record['notes']}"))
    if type7_record.get("source_query_id"):
        display(Markdown(f"**Source retrieval query:** {type7_record['source_query_id']}"))

    display(Markdown("**Aggregate Type 7 summary**"))
    display(pd.DataFrame([summary]))

    display(Markdown("**Per-query metrics for the chosen Type 7 example**"))
    metric_cols = [
        col
        for col in [
            "query_id",
            "answer_relevancy",
            "faithfulness",
            "context_precision",
            "context_recall",
        ]
        if col in selected_metrics.columns
    ]
    display(selected_metrics[metric_cols] if metric_cols else selected_metrics)

    display(Markdown("**Grounded answer**"))
    if not selected_answer.empty:
        answer_row = selected_answer.iloc[0].to_dict()
        answer_text = str(answer_row.get("answer", "")).strip()
        model_name = str(answer_row.get("model_name", ""))
        prompt_version = str(answer_row.get("prompt_version", ""))
        if model_name or prompt_version:
            display(Markdown(f"Model: `{model_name}` | Prompt: `{prompt_version}`"))
        print(answer_text if answer_text else "No answer text found.")
    else:
        print("No answer row found for the selected Type 7 query.")

    display(Markdown("**Top context rows used by RAG**"))
    context_cols = [
        col
        for col in [
            "rank",
            "doc_id",
            "caption",
            "schema",
            "meta_country",
            "meta_programId",
            "rrf_score",
        ]
        if col in selected_context.columns
    ]
    display(selected_context[context_cols] if context_cols else selected_context)

### Type 7 - Q7_001

**Query:** Summarise the sanctions-related background, key entities, and supporting evidence for: Russian oil tankers in the shadow fleet sanctioned for circumventing the price cap.

**Notes:** SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT

**Source retrieval query:** Q3_001

**Aggregate Type 7 summary**

...

**Grounded answer**

Model: `gpt-oss:20b` | Prompt: `v1`

The retrieved records identify a group of ten vessels—GRACEP, MARMEA, TIGER WINGS, ADALYNN, TRUVOR, NOBEL M, HORIZZON, URSUS ARCTOS, GALIASGAR KAMAL, and HANUMAN—that are all listed on the U.S. “UA‑WS‑MARE” sanctions programme, which is a Ukrainian‑war‑related list. Each entry includes an IMO number, call sign, and a ranking score, and all are flagged in the “ua_war_sanctions” database [Record 1‑10]. The only sanction programmes mentioned are UA‑WS‑MARE (and for NOBEL M, also US‑TERR, US‑OFAC SDN, and US‑Trade CSL) – none of the records explicitly reference the U.S. oil‑price‑cap sanctions or any activity that would indicate circumvention of that cap.

Because the records lack any mention of Russian ownership, oil cargoes, or price‑cap evasion, the evidence does not directly support the claim that these vessels belong to a “shadow fleet” of Russian oil tankers circumventing the price cap. The strongest evidence available is the inclusion of each vessel on the UA‑WS‑MARE list, which sig

**Top context rows used by RAG**

,rank,doc_id,caption,schema,meta_country,meta_programId,rrf_score
0,1,NK-gGz7BbryCSo4JZSujy7XFt,GRACEP,Vessel,,UA-WS-MARE,0.027271
1,2,NK-kKmdPdAXaFCdqrJfubLxjZ,MARMEA,Vessel,,UA-WS-MARE,0.024964
2,3,NK-LTtczqzSyDLFwNDam9VpUY,TIGER WINGS,Vessel,,UA-WS-MARE,0.024003
3,4,NK-GP4PxTt8GsMYQ3Lf3YFWpL,ADALYNN,Vessel,,UA-WS-MARE,0.023485
4,5,ua-ws-vessel-1484,TRUVOR,Vessel,,UA-WS-MARE,0.023282


## Presentation Takeaways

- **Types 1-6:** present the **RRF table** as the main system output.
- **Type 1:** exact identifiers are the cleanest architectural win because the dedicated identifier lane is purpose-built for them.
- **Types 2-5:** the comparison table helps explain why the system is hybrid rather than single-model.
- **Type 6:** if the result quality looks weaker, that is expected and is consistent with the project analysis that many of these queries behave more like metadata filters than free-text search.
- **Type 7:** present the generated answer together with the retrieved context rows to show that the narrative is grounded in evidence.

If you want to swap examples during the live demo, only change the query IDs in the config cell at the top.